In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import requests
from io import StringIO
from Bio import SeqIO
from collections import Counter

# SwissProt background (latest: Release 2025_03, 18-Jun-2025)
SWISSPROT_FREQ = {
    'A': 0.0825, 'C': 0.0138, 'D': 0.0546, 'E': 0.0671, 'F': 0.0386,
    'G': 0.0707, 'H': 0.0227, 'I': 0.0590, 'K': 0.0579, 'L': 0.0964,
    'M': 0.0241, 'N': 0.0406, 'P': 0.0474, 'Q': 0.0393, 'R': 0.0552,
    'S': 0.0665, 'T': 0.0536, 'V': 0.0685, 'W': 0.0110, 'Y': 0.0292
}

# AA groups from slide
AA_GROUPS = {
    'Apolar': ['G', 'A', 'V', 'P', 'L', 'I', 'M'],
    'Aromatic': ['F', 'W', 'Y'],
    'Polar': ['S', 'T', 'C', 'N', 'Q', 'H'],
    'Charged': ['D', 'E', 'K', 'R']
}

def load_positives(tsv_file):
    """Load TSV and filter positives."""
    df = pd.read_csv(tsv_file, sep='\t')
    df.columns = df.columns.str.strip()
    df['SP cleavage'] = pd.to_numeric(df['SP cleavage'], errors='coerce')
    df['label'] = pd.to_numeric(df['label'], errors='coerce').fillna(0).astype(int)
    positives = df[df['label'] == 1].copy()
    print(f"Loaded {len(positives)} positives from {tsv_file}")
    return positives

def fetch_sequences(accessions, batch_size=100):
    """Fetch UniProt sequences in batches."""
    sequences = {}
    for i in range(0, len(accessions), batch_size):
        batch = accessions[i:i + batch_size]
        query = '%20OR%20'.join(f'accession:{acc}' for acc in batch)
        url = f'https://rest.uniprot.org/uniprotkb/stream?format=fasta&query={query}'
        try:
            response = requests.get(url)
            response.raise_for_status()
            for record in SeqIO.parse(StringIO(response.text), 'fasta'):
                acc = record.id.split('|')[1] if '|' in record.id else record.id
                sequences[acc] = str(record.seq)
        except Exception as e:
            print(f"Batch failed: {e}")
    print(f"Fetched {len(sequences)} sequences")
    return sequences

def compute_sp_aa_freq(pos_df):
    """Compute average AA frequency in SPs."""
    aa_counts = Counter()
    total = 0
    for _, row in pos_df.iterrows():
        sp_seq = row['Sequence'][:int(row['SP cleavage'] - 1)]  # SP region
        aa_counts.update(sp_seq)
        total += len(sp_seq)
    if total == 0:
        return {}
    return {aa: count / total for aa, count in aa_counts.items()}

def plot_aa_comp(sp_freq, dataset_name):
    """Combined barplot with grouped AAs."""
    sorted_aas = [aa for group in AA_GROUPS.values() for aa in sorted(group)]
    sp_vals = [sp_freq.get(aa, 0) * 100 for aa in sorted_aas]
    swiss_vals = [SWISSPROT_FREQ.get(aa, 0) * 100 for aa in sorted_aas]
    df_plot = pd.DataFrame({
        'AA': sorted_aas * 2,
        'Frequency (%)': sp_vals + swiss_vals,
        'Group': ['SP'] * len(sorted_aas) + ['SwissProt'] * len(sorted_aas)
    })
    plt.figure(figsize=(14, 7))
    sns.barplot(x='AA', y='Frequency (%)', hue='Group', data=df_plot, palette=['#1f77b4', '#ff7f0e'])
    # Add group labels
    pos = 0
    for group, aas in AA_GROUPS.items():
        group_len = len(aas)
        plt.text(pos + group_len / 2 - 0.5, -5, group, ha='center', va='top', fontweight='bold')
        pos += group_len
    plt.title(f'AA Composition: {dataset_name} SPs vs SwissProt (Grouped)')
    plt.ylabel('Frequency (%)')
    plt.savefig(f'{dataset_name.lower()}_aa_comp.png')
    plt.close()
    print(f"Saved plot: {dataset_name.lower()}_aa_comp.png")

# Run for training
train_pos = load_positives('training_with_folds.tsv')
train_seqs = fetch_sequences(train_pos['Accession'].tolist())
train_pos['Sequence'] = train_pos['Accession'].map(train_seqs)
train_pos = train_pos.dropna(subset=['Sequence'])
train_sp_freq = compute_sp_aa_freq(train_pos)
plot_aa_comp(train_sp_freq, 'Training')

# Run for test
test_pos = load_positives('test_set_labeled.tsv')
test_seqs = fetch_sequences(test_pos['Accession'].tolist())
test_pos['Sequence'] = test_pos['Accession'].map(test_seqs)
test_pos = test_pos.dropna(subset=['Sequence'])
test_sp_freq = compute_sp_aa_freq(test_pos)
plot_aa_comp(test_sp_freq, 'Test')

Loaded 873 positives from training_with_folds.tsv
Fetched 873 sequences
Saved plot: training_aa_comp.png
Loaded 218 positives from test_set_labeled.tsv
Fetched 218 sequences
Saved plot: test_aa_comp.png
